# AI Voice Feedback Agent -- **live voice (STT + TTS)**

Same agentic flow as `voice_survey_agent.ipynb`, with the mock console engines replaced by
**real speech**: the agent's turns are spoken out loud through your speakers, and the
customer's turns are captured as audio and transcribed. Everything else -- the negotiation,
the language step, the dialogue manager, the call budget logic, the summariser -- is unchanged,
which is the point: Section 4 was always the only thing that had to change.

**Flow:**
1. Business profile input (business, target user, tone, feedback objective) -- all free text
2. LLM proposes feedback approaches and you **negotiate in plain English** until one is right
3. Language -- say it however you like ("Hindi", "brazilian portuguese"); the LLM resolves it
4. **Real STT / TTS engines** (Gemini speech, streamed) + audio device setup and a self-test
5. Real-time dialogue manager -- handles silence, off-topic questions, "who are you"
6. Run the call -- out loud
7. Structured feedback extraction from the transcript
8. Play back the recording of the whole call

**Three ways to be the customer** (`VOICE_INPUT_MODE`):

| mode | who talks | what it proves |
|---|---|---|
| `mic` | you, into your microphone | the full live loop, end to end |
| `synthetic` | a second TTS voice reading the scripted caller lines | the same real STT/TTS round trips with **no microphone needed** -- the mock call, out loud, unattended |
| `console` | you, typing | last-resort fallback when there is no audio hardware at all |

`auto` (the default) picks `mic` if a working input device is found, otherwise `synthetic`.
Colab has no local audio devices, so it lands on `synthetic` and you listen in Section 8.

**Speech engines.** Both are Gemini, so `GEMINI_API_KEY` is the only credential needed:
- **TTS** -- `gemini-3.1-flash-tts-preview`, **streamed**. Audio starts playing ~1.5s in, and each
  sentence is dispatched to the speech model the moment it closes in the turn model's JSON stream,
  so the first words are audible while the rest of the turn is still being generated.
- **STT** -- `gemini-3.5-flash-lite` over the captured WAV, with the agreed call language passed in
  as a hint. Multilingual for free, which matters once Section 3 picks something other than English.
- `STT_API_KEY` / `TTS_API_KEY` stay wired up but unused; set them and swap the two marked
  classes to move to Deepgram/ElevenLabs/Twilio without touching anything else.

**Endpointing.** A live mic has no "send" button, so `MicSTT` measures the room's noise floor
once at startup, waits for energy above it, and cuts the turn after ~1s of trailing silence --
with a hard utterance cap so one rambling answer cannot eat the whole call.

**Why the call budget is 90s here, not 45s.** On a real phone line the audio is already flowing;
in this notebook every turn additionally pays a TTS round trip (~1.5s to first audio) and an STT
round trip (~2-3s). Those are real seconds the customer would hear as dead air, so they are
counted honestly against the budget rather than hidden -- and the budget is raised so a demo call
can finish. Setting `CALL_DURATION_S = 45` still works; the call will simply cut itself short,
which is exactly what it is supposed to do. Per-turn timings are printed so you can see where the
time actually goes.

**Credentials** -- resolved from Colab Secrets, then the environment/`.env`, then a prompt:
- **Colab**: key icon in the left sidebar, secret named exactly `GEMINI_API_KEY`, notebook access on.
- **Local**: a `.env` file next to this notebook.

Use a real API key (starts with `AIza`) from [aistudio.google.com/apikey](https://aistudio.google.com/apikey).
An *ephemeral* AI Studio token (starts with `AQ.`) is short-lived and bound to the session that issued it.

In [ ]:
# Note: quote the version specs -- unquoted `>=` is a shell redirect and silently
# creates files named "0.8.0", "1.26.0", ... instead of installing anything.
# On Colab, -U matters: the preinstalled google-genai is usually too old for
# thinking_level, which is what keeps the in-call turns fast.
#
# sounddevice is what makes this notebook *audible*: it plays the streamed TTS PCM and
# captures the microphone. It needs PortAudio, which ships inside the wheel on Windows and
# macOS; on Linux/Colab add it with `!apt-get install -y libportaudio2`. If it is missing,
# or there are no audio devices at all (Colab), Section 4 degrades to a silent run that
# still does every real speech round trip and is played back in Section 8.
%pip install -q -U "google-genai>=1.0.0" "python-dotenv>=1.0.0" "sounddevice>=0.4.6" "numpy>=1.24"

## 0. Setup

In [ ]:
import json, logging, os, re, sys, time
from dataclasses import dataclass, field
from getpass import getpass
from typing import Dict, List, Optional

from dotenv import find_dotenv, load_dotenv
from google import genai
from google.genai import types
from google.genai import errors as genai_errors

IN_COLAB = "google.colab" in sys.modules

# override=True matters: load_dotenv() defaults to override=False, so once a stale
# key is in os.environ (e.g. you edited .env after the kernel started) it silently
# wins over the file and you get a confusing 401. Re-running this cell now always
# picks up the current .env without restarting the kernel.
DOTENV_PATH = find_dotenv(usecwd=True)
if DOTENV_PATH:
    load_dotenv(DOTENV_PATH, override=True)


def get_secret(name: str, required: bool = False) -> str:
    """Resolve a credential across every place it might live, so the same notebook
    runs locally and on Colab unchanged. Never hardcode keys in the notebook itself --
    they leak the moment the .ipynb is shared.

    Order: Colab Secrets (key icon in the left sidebar) -> environment/.env -> prompt.
    """
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = (userdata.get(name) or "").strip()
            if val:
                return val
        except Exception:
            pass  # secret not set, or notebook not granted access to it
    val = (os.getenv(name) or "").strip()
    if val:
        return val
    if required:
        if IN_COLAB:
            print(f"{name} not found in Colab Secrets. Add it via the key icon in the left "
                  f"sidebar (name it exactly {name} and enable notebook access), or paste it below.")
        val = getpass(f"{name}: ").strip()
        if val:
            return val
        raise RuntimeError(
            f"{name} missing. Local: put it in .env next to the notebook. "
            f"Colab: add it under Secrets (key icon) as {name}.")
    return ""


GEMINI_API_KEY = get_secret("GEMINI_API_KEY", required=True)
STT_API_KEY = get_secret("STT_API_KEY")
TTS_API_KEY = get_secret("TTS_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

# Two model tiers. In-call turns are latency-critical; the offline setup/summary
# steps are not, so they can afford the bigger model.
TURN_MODEL = "gemini-3.5-flash-lite"   # ~0.9s per turn with thinking low
# Also lite by default: the free tier caps gemini-3.5-flash at 20 requests/DAY, and one
# pass through this notebook spends several here. Switch to "gemini-3.5-flash" for
# stronger negotiation/summaries once billing is enabled. See section 0b.
PLANNING_MODEL = "gemini-3.5-flash-lite"

TEST_MODE = True   # True = no input() anywhere: canned answers + scripted customer replies
VERBOSE_LATENCY = True  # print per-turn model latency

# Thinking costs ~1s+ per turn, which is dead air on a phone call. "low" is the
# floor this model family accepts and is plenty for short scripted dialogue.
FAST_THINKING = types.ThinkingConfig(thinking_level="low")


def json_config(schema: types.Schema, max_tokens: int, system: Optional[str] = None,
                temperature: float = 0.3) -> types.GenerateContentConfig:
    """Config that forces schema-valid JSON, so no output parsing/repair is ever needed."""
    return types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        max_output_tokens=max_tokens,
        response_mime_type="application/json",
        response_schema=schema,
        thinking_config=FAST_THINKING,
    )


def with_retry(fn, attempts: int = 3):
    """Retry past rate limits. The free tier is stingy (as low as 20 requests/day on the
    bigger models), and a 429 mid-negotiation would otherwise lose the whole conversation.
    Honours the server's retryDelay when it gives one."""
    for i in range(attempts):
        try:
            return fn()
        except genai_errors.ClientError as e:
            msg = str(e)
            if getattr(e, "code", None) != 429:
                raise
            if i == attempts - 1:
                raise RuntimeError(
                    "Gemini rate limit hit and retries exhausted.\n"
                    "  The free tier allows very few requests/day on the larger models.\n"
                    "  Options: wait for the quota to reset, point PLANNING_MODEL at a "
                    "'-lite' model (they have far higher free limits), or enable billing.\n"
                    "  Check usage at https://ai.dev/rate-limit\n"
                    f"  Original error: {msg[:300]}") from None
            m = re.search(r"retryDelay['\"]?:\s*['\"]?(\d+)", msg)
            wait = min(float(m.group(1)) + 1 if m else 5 * (i + 1), 65)
            print(f"  rate limited, retrying in {wait:.0f}s...")
            time.sleep(wait)


def call_json(model: str, prompt: str, schema: types.Schema, max_tokens: int = 2048,
              system: Optional[str] = None) -> dict:
    """Single-shot JSON call with a readable error if the model output was truncated."""
    resp = with_retry(lambda: client.models.generate_content(
        model=model, contents=prompt, config=json_config(schema, max_tokens, system)))
    text = (resp.text or "").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        finish = resp.candidates[0].finish_reason if resp.candidates else "unknown"
        raise ValueError(
            f"Model returned invalid/truncated JSON (finish_reason={finish}). "
            f"Raise max_tokens for this call. Raw output: {text!r}") from e


def warm_up() -> float:
    """Preflight: pay TLS handshake + connection setup now, not on the customer's first
    turn, and fail here with an actionable message rather than mid-call."""
    t0 = time.perf_counter()
    try:
        client.models.generate_content(
            model=TURN_MODEL, contents="ok",
            config=types.GenerateContentConfig(max_output_tokens=1, thinking_config=FAST_THINKING))
    except genai_errors.ClientError as e:
        if getattr(e, "code", None) in (401, 403):
            if GEMINI_API_KEY.startswith("AQ."):
                hint = ("This is an AI Studio EPHEMERAL token ('AQ.' prefix), not an API key. It is "
                        "short-lived and tied to the session that minted it, so it commonly works on "
                        "your own machine and still fails 401 from Colab. Create a real API key "
                        "(starts with 'AIza') at https://aistudio.google.com/apikey")
            elif not GEMINI_API_KEY.startswith("AIza"):
                hint = ("A Gemini API key normally starts with 'AIza'. Get one at "
                        "https://aistudio.google.com/apikey")
            else:
                hint = ("Key looks well-formed, so it is likely revoked, or the Generative Language "
                        "API is not enabled on its project. Re-issue it at "
                        "https://aistudio.google.com/apikey")
            where = ("Colab Secrets (key icon in the left sidebar)" if IN_COLAB
                     else f".env at {DOTENV_PATH or '<none found>'}")
            raise RuntimeError(
                f"Gemini rejected the credentials ({e.code}).\n"
                f"  running on: {'Google Colab' if IN_COLAB else 'local kernel'}\n"
                f"  key source: {where}\n"
                f"  key loaded: {GEMINI_API_KEY[:6]}...{GEMINI_API_KEY[-4:]} (len {len(GEMINI_API_KEY)})\n"
                f"  {hint}\n"
                f"  After updating the key, re-run THIS cell -- it re-reads the secret/.env.") from None
        raise
    return time.perf_counter() - t0


print("Gemini configured | turn model:", TURN_MODEL, "| planning model:", PLANNING_MODEL)
print(f"Environment: {'Google Colab' if IN_COLAB else 'local kernel'}")
print(f"Key loaded: {GEMINI_API_KEY[:6]}...{GEMINI_API_KEY[-4:]} (len {len(GEMINI_API_KEY)})")
if GEMINI_API_KEY.startswith("AQ."):
    print("  WARNING: 'AQ.' is an ephemeral AI Studio token, not an API key. It expires quickly")
    print("           and is bound to its origin session -- expect 401s, especially on Colab.")
    print("           Create a durable key (starts with 'AIza') at https://aistudio.google.com/apikey")
print("STT key present:", bool(STT_API_KEY), "| TTS key present:", bool(TTS_API_KEY))
print(f"Warm-up round trip: {warm_up():.2f}s")


# --------------------------------------------------------------- voice engines
# Both speech engines are Gemini, so GEMINI_API_KEY is the only credential this
# notebook actually needs. STT_API_KEY / TTS_API_KEY stay resolved above for the
# day you swap in Deepgram/ElevenLabs/Twilio -- see Section 4.
STT_MODEL = "gemini-3.5-flash-lite"        # transcribes the captured WAV; multilingual
TTS_MODEL = "gemini-3.1-flash-tts-preview" # streams PCM out; ~1.5s to first audio
AGENT_VOICE = "Kore"    # the voice the business's agent speaks with
CALLER_VOICE = "Puck"   # only used by the synthetic caller, so the two sides sound different

TTS_SAMPLE_RATE = 24000  # fixed by the TTS model's output format
MIC_SAMPLE_RATE = 16000  # plenty for speech, and a third of the bytes to upload

# "auto" -> mic if a working input device is found, else the synthetic caller.
# Force it to "mic", "synthetic" or "console" to pin one down.
VOICE_INPUT_MODE = "auto"
PLAY_AUDIO = True        # False = still do every real round trip, just stay silent

# The SDK logs a long automatic-function-calling advisory on every generate_content
# call that carries a config; it is noise here and would bury the call transcript.
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

print("Speech  | stt:", STT_MODEL, "| tts:", TTS_MODEL, "| voice:", AGENT_VOICE)

### 0b. Which models can I use? (optional)Model names change and old ones get retired, so don't guess -- this asks your key what it can actually reach.To switch models, edit the two constants in the Setup cell above:- **`TURN_MODEL`** runs every in-call turn, so it dominates how snappy the call feels. Keep it a `-lite` / small model.- **`PLANNING_MODEL`** runs the setup negotiation and the final summary. These are off the clock, so favour capability here.If a name is retired, the API says so and names its replacement in the error, e.g. *"models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash"*.**Watch the free-tier quota.** Non-lite models have a very low free daily cap (`gemini-3.5-flash` allows **20 requests/day**), and one full pass through this notebook spends several on `PLANNING_MODEL` — the negotiation alone costs one per round. If you hit `429 RESOURCE_EXHAUSTED`, either wait for the reset, point `PLANNING_MODEL` at a `-lite` model (much higher free limits, slightly weaker negotiation), or enable billing. Live usage: [ai.dev/rate-limit](https://ai.dev/rate-limit).

In [ ]:
# Ask the API what this key can reach, rather than trusting a hardcoded list.
available = [m.name.removeprefix("models/") for m in client.models.list()
             if "generateContent" in (m.supported_actions or [])]

print(f"{len(available)} models available. Text models with 'flash' or 'pro' in the name:\n")
for name in sorted(n for n in available if ("flash" in n or "pro" in n)
                   and not any(x in n for x in ("image", "tts", "embedding", "vision"))):
    marks = []
    if name == TURN_MODEL:
        marks.append("<- TURN_MODEL")
    if name == PLANNING_MODEL:
        marks.append("<- PLANNING_MODEL")
    print(f"  {name:42} {' '.join(marks)}")

for label, chosen in (("TURN_MODEL", TURN_MODEL), ("PLANNING_MODEL", PLANNING_MODEL)):
    if chosen not in available:
        print(f"\nWARNING: {label}={chosen!r} is not in this key's model list; calls will 404.")


def benchmark(model_name: str, runs: int = 3) -> float:
    """Time a realistic short JSON turn. Use this before promoting a model to TURN_MODEL --
    anything much over ~1s is audible dead air on a phone call."""
    cfg = types.GenerateContentConfig(
        max_output_tokens=TURN_MAX_TOKENS if "TURN_MAX_TOKENS" in globals() else 160,
        temperature=0.4, thinking_config=FAST_THINKING,
        response_mime_type="application/json")
    best = float("inf")
    for _ in range(runs):
        t0 = time.perf_counter()
        client.models.generate_content(
            model=model_name, config=cfg,
            contents='Return JSON {"speech": <one short spoken sentence asking a cafe '
                     'customer to rate their visit 1 to 5>, "should_end": false}')
        best = min(best, time.perf_counter() - t0)
    return best

# Uncomment to compare candidates before switching:
# for m in ["gemini-3.5-flash-lite", "gemini-3.5-flash"]:
#     print(f"{m:28} best of 3: {benchmark(m):.2f}s")

## 1. Business Profile InputEvery field is free text typed by the business user. `TEST_MODE = True` substitutes the bracketed defaults so the whole notebook runs top to bottom without stopping for input.

In [ ]:
def ask(prompt: str, default: str = "") -> str:
    """Free-text prompt for the business user. In TEST_MODE, silently take the default."""
    if TEST_MODE:
        return default
    shown = f"{prompt} [{default}]: " if default else f"{prompt}: "
    while True:
        val = input(shown).strip()
        if val or default:
            return val or default
        print("  (this one is required)")


def ask_int(prompt: str, default: int, lo: int, hi: int) -> int:
    """Numeric prompt that re-asks instead of crashing on junk input."""
    if TEST_MODE:
        return default
    while True:
        raw = input(f"{prompt} [{default}]: ").strip()
        if not raw:
            return default
        if raw.isdigit() and lo <= int(raw) <= hi:
            return int(raw)
        print(f"  (enter a whole number from {lo} to {hi})")


business_profile = {
    "business_name": ask("Business name", "Bella Vista Cafe"),
    "business_description": ask("What does the business do?", "A neighborhood cafe serving coffee, breakfast, and lunch."),
    "target_user": ask("Who is being called?", "Walk-in customers who just finished dining or picking up an order."),
    "tone": ask("Desired call tone", "warm, friendly, concise"),
    "feedback_objective": ask("What do you want to learn from this feedback?", "Understand satisfaction with food quality and service speed, and catch any recurring complaints."),
}

for k, v in business_profile.items():
    print(f"{k:22} {v}")

## 2. Agree on a Feedback Approach (conversational)The model proposes a few feedback approaches. You are **not** picking a list index -- you reply in plain English:- `ok` / `yes` / Enter -- accept what it is proposing- *"use the second one but make it about ambience"* -- refine it- *"none of these, I want to know if they'd come back"* -- ask for something different entirelyIt keeps going until it locks in a single approach, and it decides what that approach is -- the scale, the wording, and how to score it are all its call, not a hardcoded enum.**The channel constrains what it may propose.** This is a phone call and nothing else: the customer has a phone at their ear and no screen. So every approach must be answerable by *speaking* or by *pressing a digit* (DTMF) -- never a thumbs up/down, a button, a star to tap, a link, or an SMS. Each proposal declares an `input_mode` (`speech` / `keypad` / `either`) and, when keys are involved, the mapping it will read aloud. Ask it for something the channel can't do and it will say so and offer the closest workable equivalent.Keypad answers are worth having: they survive noisy lines, heavy accents, and STT failures, which is exactly when a spoken rating gets lost.

In [ ]:
# Note there is no fixed list of dimension types. The model invents whatever scale
# fits what the business user asked for, and tells us how to score it.
#
# input_mode is the constraint that keeps proposals physically possible: this is a
# phone call, so the customer can only speak or press keys. There is no screen, so
# no thumbs up/down, no buttons, no stars to tap, no links.
APPROACH_FIELDS = {
    "name": types.Schema(type="STRING", description="Short label for this approach"),
    "description": types.Schema(type="STRING", description="One sentence, under 20 words"),
    "dimension_type": types.Schema(type="STRING", description="Machine-ish slug for the scale, e.g. rating_1_5, nps_0_10, yes_no, open_ended"),
    "input_mode": types.Schema(type="STRING", enum=["speech", "keypad", "either"],
                               description="How the customer answers: speaking, pressing digits, or either"),
    "sample_question": types.Schema(type="STRING", description="The question as spoken aloud, including the keypress instruction when input_mode is keypad"),
    "keypad_map": types.Schema(type="STRING", nullable=True,
                               description="What each digit means, e.g. '1-5 = rating, 1 worst 5 best'. Null when input_mode is speech."),
    "score_guidance": types.Schema(type="STRING", description="One sentence telling a later summarizer exactly what 'score' means here, or that it stays null"),
}
APPROACH_SCHEMA = types.Schema(type="OBJECT", properties=APPROACH_FIELDS,
                               required=["name", "description", "dimension_type",
                                         "input_mode", "sample_question", "score_guidance"])

NEGOTIATION_SCHEMA = types.Schema(
    type="OBJECT",
    properties={
        "reply": types.Schema(type="STRING", description="What to say to the business user"),
        "proposals": types.Schema(type="ARRAY", max_items=4, items=APPROACH_SCHEMA,
                                  description="Current proposals; empty once one is locked in"),
        "chosen": APPROACH_SCHEMA,  # nullable via not being required
    },
    required=["reply"])

NEGOTIATION_SYSTEM = """You are a customer-experience consultant helping a business owner design a 45-second outbound feedback call. You are talking TO the owner, not to their customer.

Business: {business_name}
Description: {business_description}
Customers being called: {target_user}
Desired call tone: {tone}
What they want to learn: {feedback_objective}

THE CHANNEL -- this constrains every idea you have
This is a live phone call and nothing else. The customer is holding a phone to their ear. They have exactly two ways to answer you:
  1. SPEAKING -- captured by speech-to-text. Good for open-ended answers and for numbers said aloud.
  2. PRESSING KEYS on the phone keypad (DTMF) -- digits 0-9, star, hash. Good for ratings and yes/no, and it survives noisy lines and heavy accents where speech recognition fails.
There is NO screen. Never propose a thumbs up or thumbs down, a button, a star to tap, a link, an emoji, a text message, an email, or a survey form. If an idea needs anything visual, it is not possible -- convert it to speech or a keypress.

Keypad rules when you use one:
- Only single digits are one keypress. A 0-10 scale cannot use "10" -- either say "press 0 for ten", or use a 1-9 range, or take that answer by speech instead.
- Always state the mapping out loud inside sample_question, e.g. "press 1 for yes, or 2 for no".
- Keep the mapping to at most 5 options; nobody remembers more while holding a phone.
- Open-ended answers must be input_mode "speech" -- a keypad cannot capture an opinion.

Your first message: propose 2-4 genuinely different approaches. Vary the input_mode across them -- at least one keypad-based and at least one spoken -- so the owner can weigh a crisp measurable answer against a richer one. Each must be answerable in under 15 seconds, and each must clearly serve what they want to learn. Keep "reply" to one short line -- the proposals are rendered separately, so do not restate them in prose.

After that, read what the owner says and act on it:
- Clear approval ("ok", "yes", "sounds good", "the second one"): set "chosen" to that exact approach and leave "proposals" empty.
- A tweak ("make it about ambience", "shorter", "use a 1 to 10 scale"): apply it. If the result is now unambiguous, put it straight in "chosen" rather than making them approve again.
- A different direction entirely: drop your earlier ideas and propose fresh ones that fit what they actually asked for.
- Something ambiguous or outside what a 45-second call can do: say so plainly in one line and offer the closest workable thing.

Rules:
- Never present the choices as a numbered list inside "reply"; they are shown from "proposals".
- sample_question must be phrased for the ear: one sentence, under 20 spoken words, no lists, digits as digits, and it must include the keypress instruction whenever input_mode is "keypad".
- If the owner asks for something the channel cannot do (a thumbs up, a star rating they can tap, a link), say plainly in one line that a phone call has no screen, then offer the closest keypad or spoken equivalent.
- Set "chosen" the moment the owner has settled on something. Do not keep asking for confirmation.
- Once "chosen" is set, "reply" is a single sentence confirming what the call will ask and how the customer answers it.
"""


MODE_LABEL = {"speech": "spoken answer", "keypad": "keypad press", "either": "spoken or keypad"}

def show_turn(turn: dict) -> None:
    print(turn["reply"])
    for p in turn.get("proposals") or []:
        print(f"\n  - {p['name']} ({p['dimension_type']}, {MODE_LABEL.get(p['input_mode'], p['input_mode'])})")
        print(f"    {p['description']}")
        print(f"    asks: \"{p['sample_question']}\"")
        if p.get("keypad_map"):
            print(f"    keys: {p['keypad_map']}")


def negotiate_approach(profile: dict, max_rounds: int = 8) -> dict:
    """Talk to the model in plain English until a single approach is agreed on."""
    chat = client.chats.create(
        model=PLANNING_MODEL,
        config=json_config(NEGOTIATION_SCHEMA, 2048,
                           system=NEGOTIATION_SYSTEM.format(**profile), temperature=0.6))

    # In TEST_MODE, exercise the refine path (not just instant approval) so the
    # negotiation is actually demonstrated end to end. The middle turn asks for
    # something the channel cannot do, to check the model pushes back instead of
    # cheerfully agreeing to a thumbs-up button on a phone call.
    scripted = ["Can they just give a thumbs up or thumbs down?",
                "Alright. What I really care about is whether they'd come back -- ask that.",
                "Good, use the keypad one."]

    def send(msg: str) -> dict:
        return json.loads(with_retry(lambda: chat.send_message(msg)).text)

    turn = send("Propose some approaches.")
    for _ in range(max_rounds):
        show_turn(turn)
        if turn.get("chosen"):
            return turn["chosen"]
        if TEST_MODE:
            if not scripted:
                raise RuntimeError("TEST_MODE script ran out before an approach was agreed.")
            reply = scripted.pop(0)
            print(f"\n> {reply}")
        else:
            reply = input("\n> ").strip() or "Yes, that works. Use it."
        turn = send(reply)

    raise RuntimeError("Could not settle on an approach; restart this cell and be more specific.")


selected_strategy = negotiate_approach(business_profile)
print("\n" + "=" * 60)
print("Locked in:", selected_strategy["name"], f"({selected_strategy['dimension_type']})")
print("Answered :", MODE_LABEL.get(selected_strategy["input_mode"], selected_strategy["input_mode"]))
print("Will ask :", selected_strategy["sample_question"])
if selected_strategy.get("keypad_map"):
    print("Keys     :", selected_strategy["keypad_map"])
print("Scoring  :", selected_strategy["score_guidance"])

## 3. Call LanguageType it however you like -- "Hindi", "brazilian portuguese", "Spanish but casual". The model resolves it to a language name and BCP-47 code for the STT/TTS engines, and writes the fallback goodbye line in that language now, so a hard cutoff mid-call never has to wait on a model round trip.

In [ ]:
LANGUAGE_SCHEMA = types.Schema(
    type="OBJECT",
    properties={
        "language_name": types.Schema(type="STRING", description="English name of the language, e.g. 'Hindi'"),
        "language_code": types.Schema(type="STRING", description="BCP-47 code for STT/TTS, e.g. 'hi' or 'pt-BR'"),
        "closing_line": types.Schema(type="STRING", description="A warm 10-word goodbye IN that language, in its native script"),
    },
    required=["language_name", "language_code", "closing_line"])

LANGUAGE_PROMPT = """The business owner wants the feedback call conducted in: "{raw}"

Resolve that to a language for a speech engine. Honour regional intent (e.g. "brazilian portuguese" -> pt-BR).
closing_line must be a natural spoken sign-off thanking the customer for their time, written in that language's
native script, under 10 words. If the request is unclear or not a language, fall back to English."""

language_request = ask("What language should the call be in?", "English")
CALL_LANGUAGE = call_json(PLANNING_MODEL, LANGUAGE_PROMPT.format(raw=language_request),
                          LANGUAGE_SCHEMA, max_tokens=512)

LANG_CODE = CALL_LANGUAGE["language_code"]
LANG_NAME = CALL_LANGUAGE["language_name"]
FALLBACK_CLOSING = CALL_LANGUAGE["closing_line"]

print(f"Call language: {LANG_NAME} ({LANG_CODE})")
print(f"Fallback closing: {FALLBACK_CLOSING}")

## 3b. Feedback DepthHow many feedback questions the agent may ask inside the 45s. Two is the practical ceiling for a call this short.

In [ ]:
MAX_QUESTIONS = ask_int("Max feedback questions to ask in the call (1 or 2)", 3, 1, 3)
print(f"Max feedback questions per call: {MAX_QUESTIONS}")

## 4. STT / TTS Engines (real speech)

This is the only section that differs from the mock notebook. The engines below implement the
same two-method contract (`listen()` / `synthesize()`), so the dialogue manager in Section 5 and
everything after it are identical to the console version.

**`GeminiTTS.synthesize()` -- streamed, twice over.** The dialogue manager already calls it with
*growing* text as the turn model's JSON streams in. Rather than buffer the turn, it dispatches
each sentence to the speech model the moment that sentence closes, on a background worker, so
audio for sentence one is playing while sentence two is still being generated. `final=True`
flushes the remainder and blocks until the speaker has actually finished -- the call must not
start listening while the agent is still talking.

**`MicSTT.listen()` -- endpointing, because a phone has no send button.** The noise floor is
measured once at startup; a turn starts when energy crosses it and ends after `SILENCE_TAIL_S`
of quiet, with a hard cap so one rambling answer cannot consume the whole call. The captured PCM
goes to Gemini as a WAV with the call language as a hint.

**`SyntheticCallerSTT` -- the mock call, out loud, with no microphone.** It speaks each scripted
customer line in a second voice and then puts that audio through the *real* STT path. Every round
trip is genuine; only the human is simulated. This is what runs on Colab, and it is the right
thing to run in CI.

**DTMF.** There is no phone line here, so there are no real keypresses. A transcript that is
nothing but digits ("4") is treated as a keypress so the keypad path stays exercisable, and the
system prompt already tells the agent to accept a spoken number just as readily. On a real leg,
`digits` comes from your telephony provider's DTMF/gather events (Twilio, Vonage, Plivo all emit
them) and this heuristic goes away.

**Everything both sides say is recorded** into one mono timeline, so Section 8 can play the whole
call back.

### 4a. Audio devices

In [ ]:
import io, queue, threading, wave

try:
    import numpy as np
    import sounddevice as sd
    AUDIO_OK, AUDIO_ERR = True, ""
except Exception as e:            # no PortAudio (common on Linux/Colab), no numpy, ...
    AUDIO_OK, AUDIO_ERR = False, f"{type(e).__name__}: {e}"


def audio_devices() -> tuple:
    """(inputs, outputs) as [(index, name), ...]. Empty lists on a machine with no audio."""
    if not AUDIO_OK:
        return [], []
    ins, outs = [], []
    for i, d in enumerate(sd.query_devices()):
        if d["max_input_channels"] > 0:
            ins.append((i, d["name"]))
        if d["max_output_channels"] > 0:
            outs.append((i, d["name"]))
    return ins, outs


INPUT_DEVICE = None    # None = let the block below choose; set an index from the printout
OUTPUT_DEVICE = None

_inputs, _outputs = audio_devices()
if AUDIO_OK:
    print(f"{len(_inputs)} input / {len(_outputs)} output device(s)")
    for i, name in _inputs[:6]:
        print(f"  in  [{i:2}] {name}")
    for i, name in _outputs[:6]:
        print(f"  out [{i:2}] {name}")
else:
    print("sounddevice unavailable ->", AUDIO_ERR)
    print("  The call still runs and is recorded; listen to it in Section 8.")

# Loopback devices record what the speakers are playing. Auto-selecting one would make
# the agent transcribe *itself* -- a genuinely confusing bug to chase -- so they are only
# ever used if you name the index explicitly.
LOOPBACK_HINTS = ("stereo mix", "what u hear", "loopback", "wave out", "line in")

# sd.default.device comes back as (-1, n) on machines where the OS never assigned a
# default *recording* device -- a very common state, and one that makes InputStream
# fail with a confusing error. Pick an explicit index instead of trusting the default.
if AUDIO_OK and INPUT_DEVICE is None:
    default_in = sd.default.device[0] if isinstance(sd.default.device, (list, tuple)) else None
    real_mics = [(i, n) for i, n in _inputs
                 if not any(h in n.lower() for h in LOOPBACK_HINTS)]
    preferred = [i for i, n in real_mics if "mic" in n.lower()] or [i for i, _ in real_mics]
    INPUT_DEVICE = (default_in if isinstance(default_in, int) and default_in >= 0
                    else (preferred[0] if preferred else None))
    if INPUT_DEVICE is None and _inputs:
        print("  Only loopback inputs found; set INPUT_DEVICE by hand to use one.")
if AUDIO_OK and OUTPUT_DEVICE is None:
    default_out = sd.default.device[1] if isinstance(sd.default.device, (list, tuple)) else None
    OUTPUT_DEVICE = default_out if isinstance(default_out, int) and default_out >= 0 else (
        _outputs[0][0] if _outputs else None)

print("Chosen  | input:", INPUT_DEVICE, "| output:", OUTPUT_DEVICE)

### 4b. Capture, playback and recording

In [ ]:
# ---------------------------------------------------------------- audio plumbing
# Playback, capture and recording, kept apart from the engines so swapping in a
# third-party STT/TTS vendor later touches only the engine classes below.

SILENCE_TAIL_S = 1.0     # quiet this long ends the customer's turn
MIN_UTTERANCE_S = 0.35   # shorter than this is a cough or a door, not an answer
MAX_UTTERANCE_S = 12.0   # hard cap: one rambling answer must not eat the whole call
NOISE_FLOOR = 0.004      # RMS; re-measured for the real room by calibrate_noise_floor()
VOICE_MARGIN = 3.5       # speech must be this many times the noise floor to count


def pcm_to_wav(pcm: bytes, samplerate: int) -> bytes:
    """Wrap raw mono 16-bit PCM in a WAV container (what the STT model wants)."""
    buf = io.BytesIO()
    with wave.open(buf, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(samplerate)
        w.writeframes(pcm)
    return buf.getvalue()


class CallRecorder:
    """One mono timeline of everything either side said, in order, for Section 8.
    Mic audio is 16k and TTS is 24k, so the odd one out is resampled on the way in."""

    def __init__(self, samplerate: int = TTS_SAMPLE_RATE):
        self.rate = samplerate
        self._chunks: List[bytes] = []

    def add(self, pcm: bytes, samplerate: int) -> None:
        if not pcm:
            return
        if samplerate != self.rate and AUDIO_OK:
            src = np.frombuffer(pcm, dtype=np.int16).astype(np.float32)
            n = max(1, int(len(src) * self.rate / samplerate))
            dst = np.interp(np.linspace(0, len(src) - 1, n), np.arange(len(src)), src)
            pcm = dst.astype(np.int16).tobytes()
        self._chunks.append(pcm)

    def add_silence(self, seconds: float) -> None:
        self._chunks.append(b"\x00\x00" * int(self.rate * seconds))

    @property
    def seconds(self) -> float:
        return sum(len(c) for c in self._chunks) / 2 / self.rate

    def wav_bytes(self) -> bytes:
        return pcm_to_wav(b"".join(self._chunks), self.rate)


class Speaker:
    """Sequential PCM playback. One output stream stays open for the whole call, so chunks
    butt up against each other instead of clicking, and write() blocking is what guarantees
    the agent finishes a sentence before the next one starts."""

    def __init__(self, recorder: CallRecorder, samplerate: int = TTS_SAMPLE_RATE,
                 device=None, enabled: bool = True):
        self.recorder = recorder
        self.rate = samplerate
        self.device = device
        self.enabled = enabled and AUDIO_OK and device is not None
        self._stream = None

    def _ensure(self):
        if self._stream is None and self.enabled:
            self._stream = sd.OutputStream(samplerate=self.rate, channels=1,
                                           dtype="int16", device=self.device)
            self._stream.start()
        return self._stream

    def play(self, pcm: bytes) -> None:
        self.recorder.add(pcm, self.rate)     # recorded whether or not it is audible
        if not pcm or not self.enabled:
            return
        try:
            self._ensure().write(np.frombuffer(pcm, dtype=np.int16).reshape(-1, 1))
        except Exception as e:
            # A dead output device must not take the call down with it.
            print(f"\n[audio] playback disabled ({type(e).__name__}: {e})")
            self.enabled = False

    def close(self) -> None:
        if self._stream is not None:
            try:
                self._stream.stop()
                self._stream.close()
            finally:
                self._stream = None


def calibrate_noise_floor(seconds: float = 0.8) -> float:
    """Measure the room. A fixed threshold is wrong everywhere -- a quiet room sits near
    0.001 RMS and a laptop fan near 0.02 -- and getting it wrong means the agent either
    never hears the customer or never stops listening."""
    global NOISE_FLOOR
    if not (AUDIO_OK and INPUT_DEVICE is not None):
        return NOISE_FLOOR
    try:
        rec = sd.rec(int(seconds * MIC_SAMPLE_RATE), samplerate=MIC_SAMPLE_RATE,
                     channels=1, dtype="int16", device=INPUT_DEVICE)
        sd.wait()
        rms = float(np.sqrt(np.mean((rec.astype(np.float32) / 32768.0) ** 2)))
        NOISE_FLOOR = max(rms, 0.002)
    except Exception as e:
        print(f"[audio] noise calibration failed ({type(e).__name__}: {e}); keeping default")
    return NOISE_FLOOR


def record_utterance(wait_s: float, device=None) -> tuple:
    """Listen for one customer turn. Returns (pcm_bytes, seconds_of_audio).

    Endpointing, because a phone call has no send button:
      - wait up to `wait_s` for the caller to start speaking (returns b"" if they never do)
      - keep a short pre-roll so the first syllable is not clipped
      - stop after SILENCE_TAIL_S of quiet, or at MAX_UTTERANCE_S, whichever comes first
    """
    threshold = NOISE_FLOOR * VOICE_MARGIN
    block = int(MIC_SAMPLE_RATE * 0.03)
    pre_roll = 8                      # ~240ms kept from before speech was detected
    q: "queue.Queue" = queue.Queue()

    def cb(indata, frames, t, status):
        q.put(bytes(indata))

    frames: List[bytes] = []
    started = False
    t0 = time.perf_counter()
    last_voice = t0
    with sd.InputStream(samplerate=MIC_SAMPLE_RATE, channels=1, dtype="int16",
                        blocksize=block, device=device, callback=cb):
        while True:
            try:
                buf = q.get(timeout=0.15)
            except queue.Empty:
                buf = None
            now = time.perf_counter()
            if buf is not None:
                samples = np.frombuffer(buf, dtype=np.int16).astype(np.float32) / 32768.0
                if float(np.sqrt(np.mean(samples ** 2))) > threshold:
                    if not started:
                        started = True
                        frames = frames[-pre_roll:]
                    last_voice = now
                if started:
                    frames.append(buf)
                else:
                    frames = (frames + [buf])[-pre_roll:]
            if not started and now - t0 > wait_s:
                return b"", 0.0
            if started and (now - last_voice > SILENCE_TAIL_S or now - t0 > MAX_UTTERANCE_S):
                break

    pcm = b"".join(frames)
    return pcm, len(pcm) / 2 / MIC_SAMPLE_RATE

### 4c. The engine contract

Unchanged from the mock notebook: this is the interface the dialogue manager codes against, and the reason swapping console engines for real speech touches nothing downstream.

In [ ]:
@dataclass
class CustomerInput:
    """What came back from the caller on one turn. A phone gives us exactly two
    channels, and a real telephony leg can deliver either at any moment -- a caller
    may press a key while the prompt is still playing, or just answer out loud."""
    text: str = ""     # transcribed speech
    digits: str = ""   # DTMF keypresses, e.g. "4"

    @property
    def empty(self) -> bool:
        return not (self.text or self.digits)

    def as_model_message(self) -> str:
        """How this turn is described to the model."""
        if self.digits and self.text:
            return f'<keypad: {self.digits}> {self.text}'
        if self.digits:
            return f'<keypad: {self.digits}>'
        return self.text or "<no response>"

    def as_transcript(self) -> str:
        if self.empty:
            return "<silence>"
        if self.digits and self.text:
            return f'[pressed {self.digits}] {self.text}'
        if self.digits:
            return f'[pressed {self.digits}]'
        return self.text


class STTEngine:
    def listen(self, language: str, timeout_s: float, expect_digits: bool) -> CustomerInput:
        """Listen for up to timeout_s seconds. Return speech, DTMF digits, or neither.
        `expect_digits` is a hint that the current question asked for a keypress -- a real
        implementation should still accept speech, since callers ignore instructions."""
        raise NotImplementedError

    def billable_seconds(self) -> float:
        """Wall-clock seconds of the last listen() that should count against the
        call budget. Real engines: all of it. Console mock: none (human typing time)."""
        return 0.0


class TTSEngine:
    def synthesize(self, text: str, language: str, final: bool = True) -> None:
        """Speak `text`. Called repeatedly with growing text as the model streams;
        `final=True` marks the last chunk of this turn."""
        raise NotImplementedError

### 4d. The engines

In [ ]:
# ------------------------------------------------------------------ the engines
# Same two-method contract as the console mock, so nothing downstream changes.

TTS_STYLE = "a natural, {tone} customer-service voice on a phone call"
_SENTENCE_END = re.compile(r'[.!?…。！？।]+["\')\]]?\s')

STT_PROMPT = """This is one turn of audio from a customer on a phone call, spoken in {language}.
Transcribe exactly what the person says, verbatim, in that language's own script.
Return ONLY the transcript text -- no quotes, no speaker labels, no commentary, no description of
background noise. If nobody speaks, or the audio is only noise or silence, return exactly: <none>"""


def _inline_audio(chunk) -> bytes:
    """Pull raw PCM out of a streamed TTS chunk, tolerating metadata-only chunks."""
    try:
        return chunk.candidates[0].content.parts[0].inline_data.data or b""
    except (AttributeError, IndexError, TypeError):
        return b""


def speak_stream(text: str, language: str, voice: str, style: str, on_audio) -> float:
    """Stream one utterance out of the TTS model, handing each PCM chunk to `on_audio` as
    it lands. Returns seconds to first audio -- the number that decides whether the agent
    sounds responsive or laggy, so it is measured on every turn rather than assumed."""
    cfg = types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(voice_config=types.VoiceConfig(
            prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=voice))))
    prompt = f"Read this aloud in {language} in {style}. Read only the words, nothing else:\n{text}"
    t0 = time.perf_counter()
    first = 0.0
    # with_retry needs the call to raise inside it, so the stream is materialised here
    # rather than lazily during iteration.
    for chunk in with_retry(lambda: list(client.models.generate_content_stream(
            model=TTS_MODEL, contents=prompt, config=cfg))):
        pcm = _inline_audio(chunk)
        if not pcm:
            continue
        if not first:
            first = time.perf_counter() - t0
        on_audio(pcm)
    return first


def transcribe(wav_bytes: bytes, language_name: str) -> str:
    """One STT call. Swap for Deepgram/Whisper/Twilio using STT_API_KEY; the contract is
    just bytes in, text out."""
    resp = with_retry(lambda: client.models.generate_content(
        model=STT_MODEL,
        contents=[types.Part.from_bytes(data=wav_bytes, mime_type="audio/wav"),
                  STT_PROMPT.format(language=language_name)],
        config=types.GenerateContentConfig(
            temperature=0.0, max_output_tokens=256, thinking_config=FAST_THINKING)))
    text = (resp.text or "").strip().strip('"')
    # The model occasionally narrates silence instead of returning the sentinel.
    if text.lower() in ("<none>", "none", "", "(silence)", "[silence]", "<no speech>"):
        return ""
    return text


def to_customer_input(text: str) -> CustomerInput:
    """Digits-only speech stands in for a keypress, so the keypad path stays exercisable
    without a phone line. On a real leg this heuristic is replaced by true DTMF events."""
    bare = re.sub(r"[\s.,!?]", "", text)
    if bare and all(c in "0123456789*#" for c in bare):
        return CustomerInput(digits=bare)
    return CustomerInput(text=text)


class GeminiTTS(TTSEngine):
    """Streaming text-to-speech. Replace with ElevenLabs/Google Cloud TTS using TTS_API_KEY
    and nothing else in the notebook changes.

    The dialogue manager hands us the reply as it grows. We dispatch each sentence to the
    speech model the moment it closes, on a worker thread, so sentence one is already
    playing while sentence two is still being generated by the turn model."""

    def __init__(self, speaker: "Speaker", tone: str = "warm, friendly, concise",
                 voice: str = AGENT_VOICE):
        self.speaker, self.voice = speaker, voice
        self.style = TTS_STYLE.format(tone=tone)
        self._printed = 0      # chars echoed to the console
        self._sent = 0         # chars already handed to the speech model
        self.first_audio_s = 0.0
        self.speaking_s = 0.0
        self._q: "queue.Queue" = queue.Queue()
        threading.Thread(target=self._run, daemon=True).start()

    def _run(self):
        while True:
            item = self._q.get()
            try:
                if item is None:
                    return
                text, language = item
                t0 = time.perf_counter()
                first = speak_stream(text, language, self.voice, self.style, self.speaker.play)
                self.first_audio_s = self.first_audio_s or first
                self.speaking_s += time.perf_counter() - t0
            except Exception as e:
                # Losing audio on one sentence is survivable; the text is already on screen.
                print(f"\n[tts] {type(e).__name__}: {str(e)[:160]}")
            finally:
                self._q.task_done()

    def synthesize(self, text: str, language: str, final: bool = True) -> None:
        if self._printed == 0 and text:
            print(f"[AI ({language})]: ", end="", flush=True)
        if len(text) > self._printed:                  # echo words as they arrive
            print(text[self._printed:], end="", flush=True)
            self._printed = len(text)

        pending = text[self._sent:]
        if final:
            if pending.strip():
                self._q.put((pending.strip(), language))
            self._q.join()          # the agent must finish talking before we listen
            self._printed = self._sent = 0
            print(flush=True)
            if VERBOSE_LATENCY and self.speaking_s:
                print(f"    (tts first audio {self.first_audio_s:.2f}s, "
                      f"speech {self.speaking_s:.2f}s)")
            self.first_audio_s = self.speaking_s = 0.0
        else:
            cuts = list(_SENTENCE_END.finditer(pending))
            if cuts:                                   # a sentence closed: say it now
                cut = cuts[-1].end()
                self._q.put((pending[:cut].strip(), language))
                self._sent += cut

    def close(self) -> None:
        self._q.put(None)


class MicSTT(STTEngine):
    """Live microphone capture with silence endpointing, then Gemini transcription."""

    def __init__(self, recorder: CallRecorder, device=None):
        self.recorder, self.device = recorder, device
        self._elapsed = 0.0

    def listen(self, language: str, timeout_s: float, expect_digits: bool) -> CustomerInput:
        t0 = time.perf_counter()
        hint = "speak, or say the number" if expect_digits else "speak your answer"
        print(f"[listening -- {hint}]", flush=True)
        pcm, secs = record_utterance(max(2.0, timeout_s), device=self.device)
        self.recorder.add(pcm, MIC_SAMPLE_RATE)

        if secs < MIN_UTTERANCE_S:
            self._elapsed = time.perf_counter() - t0
            print("[customer] <silence>")
            return CustomerInput()

        t1 = time.perf_counter()
        text = transcribe(pcm_to_wav(pcm, MIC_SAMPLE_RATE), language)
        stt_s = time.perf_counter() - t1
        self._elapsed = time.perf_counter() - t0
        result = to_customer_input(text)
        print(f"[customer] {result.as_transcript()}")
        if VERBOSE_LATENCY:
            print(f"    (heard {secs:.1f}s of audio, stt {stt_s:.2f}s)")
        return result

    def billable_seconds(self) -> float:
        # A real leg is live audio the whole time, so all of this is airtime.
        return self._elapsed


class SyntheticCallerSTT(STTEngine):
    """The scripted customer, spoken out loud in a second voice and then put through the
    real STT path. No microphone needed and every round trip genuine -- this is what runs
    on Colab, and what you would run in CI."""

    def __init__(self, lines: List[str], speaker: "Speaker", recorder: CallRecorder,
                 voice: str = CALLER_VOICE):
        self.lines = list(lines)
        self.speaker, self.recorder, self.voice = speaker, recorder, voice
        self._elapsed = 0.0

    def listen(self, language: str, timeout_s: float, expect_digits: bool) -> CustomerInput:
        t0 = time.perf_counter()
        line = self.lines.pop(0) if self.lines else ""
        if not line.strip():
            # Real silence: hold the line open the way a real caller's pause would.
            time.sleep(min(1.5, max(0.5, timeout_s / 4)))
            self.recorder.add_silence(1.0)
            self._elapsed = time.perf_counter() - t0
            print("[customer] <silence>")
            return CustomerInput()

        audio = bytearray()

        def sink(pcm: bytes) -> None:
            audio.extend(pcm)
            self.speaker.play(pcm)   # audible, and recorded into the call timeline

        speak_stream(line, language, self.voice,
                     "a casual, unhurried customer answering a phone call", sink)
        t1 = time.perf_counter()
        # Transcribe what was actually spoken, not the script -- this is a real STT round
        # trip, so a transcription failure shows up here instead of being papered over.
        text = transcribe(pcm_to_wav(bytes(audio), TTS_SAMPLE_RATE), language)
        stt_s = time.perf_counter() - t1
        self._elapsed = time.perf_counter() - t0
        result = to_customer_input(text or line)
        print(f"[customer] {result.as_transcript()}")
        if VERBOSE_LATENCY:
            print(f"    (spoke {len(audio) / 2 / TTS_SAMPLE_RATE:.1f}s, stt {stt_s:.2f}s)")
        return result

    def billable_seconds(self) -> float:
        return self._elapsed


class ConsoleSTT(STTEngine):
    """Typed fallback, identical to the mock notebook's engine. Only reached when there is
    no audio hardware and you explicitly asked not to use the synthetic caller."""

    def __init__(self, scripted: Optional[List[str]] = None):
        self.scripted = list(scripted or [])

    def listen(self, language: str, timeout_s: float, expect_digits: bool) -> CustomerInput:
        if self.scripted:
            raw = self.scripted.pop(0)
        else:
            raw = input(f"[customer -- {'press keys or speak' if expect_digits else 'speak'}] ")
        result = to_customer_input(raw.strip())
        print(f"[customer] {result.as_transcript()}")
        return result

    def billable_seconds(self) -> float:
        return 0.0   # human typing time is not airtime


# Scripted customer: an identity question, an off-topic detour, a keypress answer, a spoken
# elaboration, and silence -- every branch the dialogue manager handles.
SCRIPTED_CUSTOMER = [
    "Wait, who is this exactly?",
    "Okay. By the way, what time do you close today?",
    "4",
    "The food was great but I waited a while.",
    "",
]

### 4e. Build the engines

In [ ]:
# Decide how the customer speaks, build the engines, and warm the speech models up so
# the first turn of the call is not the one paying for the TLS handshake.

call_recorder = CallRecorder()
speaker = Speaker(call_recorder, device=OUTPUT_DEVICE, enabled=PLAY_AUDIO)

mic_available = AUDIO_OK and INPUT_DEVICE is not None
resolved_mode = VOICE_INPUT_MODE
if resolved_mode == "auto":
    resolved_mode = "mic" if mic_available else "synthetic"
if resolved_mode == "mic" and not mic_available:
    print("No usable input device; falling back to the synthetic caller.")
    resolved_mode = "synthetic"

if resolved_mode == "mic":
    print(f"Calibrating noise floor on device {INPUT_DEVICE} (stay quiet)...", end=" ", flush=True)
    print(f"RMS {calibrate_noise_floor():.4f} -> speech threshold {NOISE_FLOOR * VOICE_MARGIN:.4f}")
    stt_engine: STTEngine = MicSTT(call_recorder, device=INPUT_DEVICE)
elif resolved_mode == "synthetic":
    stt_engine = SyntheticCallerSTT(SCRIPTED_CUSTOMER, speaker, call_recorder)
else:
    stt_engine = ConsoleSTT(SCRIPTED_CUSTOMER if TEST_MODE else None)

tts_engine = GeminiTTS(speaker, tone=business_profile["tone"])
VOICE_INPUT_MODE_RESOLVED = resolved_mode

print(f"Engines ready: {type(stt_engine).__name__} / {type(tts_engine).__name__}")
print(f"Customer side: {resolved_mode} | playback: {'on' if speaker.enabled else 'off (silent run)'}")
if not speaker.enabled:
    print("  No output device -- the call still runs and is recorded; play it back in Section 8.")

### 4f. Audio self-test

In [ ]:
# Self-test: prove the audio path works BEFORE the timed call, where a dead speaker or a
# muted mic would just look like a customer who never answered.
RUN_AUDIO_SELFTEST = True

if RUN_AUDIO_SELFTEST:
    t0 = time.perf_counter()
    tts_engine.synthesize("Audio check. This is how the feedback call will sound.",
                          LANG_NAME, final=True)
    print(f"TTS ok: {time.perf_counter() - t0:.2f}s for one sentence, "
          f"{call_recorder.seconds:.1f}s of audio recorded so far")

    if VOICE_INPUT_MODE_RESOLVED == "mic":
        print("\nSay something short (it stops on its own after a second of silence):")
        pcm, secs = record_utterance(6.0, device=INPUT_DEVICE)
        if secs < MIN_UTTERANCE_S:
            print("  heard nothing usable. Check the mic and the INPUT_DEVICE index, lower "
                  "VOICE_MARGIN if you speak quietly, or set VOICE_INPUT_MODE = 'synthetic' "
                  "and re-run the engines cell.")
        else:
            t1 = time.perf_counter()
            heard = transcribe(pcm_to_wav(pcm, MIC_SAMPLE_RATE), LANG_NAME)
            print(f"  heard {secs:.1f}s -> {heard!r} (stt {time.perf_counter() - t1:.2f}s)")
    else:
        print(f"\nCustomer side is '{VOICE_INPUT_MODE_RESOLVED}' -- no microphone involved.")
else:
    print("Self-test skipped.")

## 5. Dialogue ManagerRuns the actual call: builds a system prompt from the business profile + agreed approach + language, then drives a turn-by-turn Gemini conversation inside a **hard 45-second budget**. Handles silence, off-topic/irrelevant questions, and identity questions ("who are you") in real time via the system prompt, and force-ends the call when time runs out.The reply schema is enforced by the API (`response_schema`), so the prompt spends its words on *behavior* instead of on describing a JSON shape the model might still get wrong.

In [ ]:
# Real speech costs real seconds: every turn pays a TTS round trip (~1.5s to first
# audio) plus an STT round trip (~2-3s) on top of the model turn, and all of it is
# airtime the customer would hear. None of it is hidden from the budget, so the budget
# itself is raised to let a demo call finish. 45 still works -- the call just ends early
# on time_up, which is the correct behaviour, not a bug.
CALL_DURATION_S = 90
TURN_LISTEN_TIMEOUT_S = 10   # how long to wait for the caller to START speaking
MAX_SILENCE_STRIKES = 2      # 2 silences -> re-prompt, 3rd -> hang up
MAX_TURNS = 8                # hard safety cap regardless of the timer
TURN_MAX_TOKENS = 160        # a spoken turn is ~25 words; keeps generation fast

END_REASONS = ["completed", "silence", "off_topic", "hostile", "declined"]

TURN_SCHEMA = types.Schema(
    type="OBJECT",
    properties={
        "speech": types.Schema(type="STRING"),
        "should_end": types.Schema(type="BOOLEAN"),
        "end_reason": types.Schema(type="STRING", nullable=True, enum=END_REASONS),
        "score": types.Schema(type="NUMBER", nullable=True),
        "sentiment": types.Schema(type="STRING", nullable=True, enum=["positive", "neutral", "negative"]),
    },

    # `speech` first is deliberate: the schema order is the generation order, so the
    # spoken text streams out before the metadata fields and TTS can start early.
    required=["speech", "should_end"])

SYSTEM_PROMPT_TEMPLATE = """You are a voice agent phoning a customer of "{business_name}" for feedback.
Tone: {tone}. Speak only {language_name}.

Approach: {strategy_name} ({dimension_type}) -- {strategy_description}
Question to work from: "{sample_question}"
How the customer answers: {input_mode_line}
What the business wants to learn: {feedback_objective}

THE CHANNEL
You are a voice on a phone. The customer has no screen, so never refer to anything they could look at, tap, or click, and never mention a thumbs up, a button, a link, or a text message. They can only speak or press keys.
A message like "<keypad: 4>" means they pressed 4 on their phone. Treat that as their answer -- acknowledge it naturally in words ("Got it, a 4") and never ask them to repeat it out loud.
If they speak a number when you asked for a keypress, or press a key when you asked them to speak, accept it either way. Never correct them about how they answered.
If they press a key that is not in the mapping you gave, restate the valid keys once, briefly.

CALL SHAPE
- Turn 1: one short greeting clause naming the business, then your first feedback question. No small talk.
- Ask AT MOST {max_questions} feedback question(s) all call. A silence re-prompt or a one-line redirect is not a new question.
- The instant you have answers to {max_questions} question(s), thank them and end. Never keep the line open to be polite.
- The whole call must fit in {call_seconds} seconds, so every reply is at most 2 sentences and under 25 spoken words.

VOICE STYLE
- Write words a person says out loud: no markdown, no bullet points, no emoji, no stage directions.
- Say digits as digits ("rate us 1 to 5"). Never read the approach name or any internal label aloud.
- When the question needs a keypress, say the mapping plainly and briefly ("press 1 for yes, 2 for no"), and say it only once unless they press something invalid.
- Do not repeat a question the customer already answered, and never re-introduce yourself twice.

HANDLING THE CUSTOMER
- "<no response>" means silence, unintelligible audio, or no keypress. Re-prompt once, shorter and simpler -- if the question had a keypad option, lead with that on the retry, since it works when the line is noisy. On a second silence, say a brief goodbye and set should_end with end_reason "silence".
- Asked who you are or how this works: answer in one short clause ("an automated feedback call for {business_name}"), then immediately re-ask your question in the same turn.
- Off-topic (hours, prices, an order problem, wants a human, small talk): you do NOT know any business fact beyond what is written above, so never answer the question itself. Say you do not have that detail but a team member can follow up, then re-ask in the same turn. If they derail a second time, end with end_reason "off_topic".
- Hostile, abusive, or a clear refusal: apologize once, thank them, end immediately with end_reason "hostile" or "declined".
- A vague answer ("it was fine") to a scored question: ask once for the number, and only once.
- You have no access to hours, prices, menu, staff, or this customer's order. Saying "I don't have that in front of me" is always correct; guessing never is. Never promise a discount, refund, or callback time.
- Anything unexpected: use your best judgment in the spirit of these rules.

FIELDS
- should_end true only when the speech you are returning IS the goodbye. Set end_reason only then.
- score: {score_guidance} Report it ONLY once the customer has actually given that answer -- a valid keypress, or the number/answer spoken aloud. An invalid keypress, a vague remark, or a question from them is not an answer: leave score null. Never guess a score to fill the field. Once they have genuinely answered, repeat that same score on every later turn.
- sentiment reflects the customer's tone so far, or null before they have said anything substantive.
"""


@dataclass
class CallResult:
    transcript: List[Dict[str, str]] = field(default_factory=list)
    end_reason: str = ""
    score: Optional[float] = None
    sentiment: Optional[str] = None
    keypresses: List[str] = field(default_factory=list)
    turn_latencies_s: List[float] = field(default_factory=list)
    duration_s: float = 0.0


INPUT_MODE_LINES = {
    "speech": "By speaking. Do not ask for a keypress.",
    "keypad": "By pressing a key on the phone keypad. {keys} State the mapping when you ask.",
    "either": "Either by speaking or by pressing a key. {keys} Offer the keypress option, but accept a spoken answer just as readily.",
}


def build_system_prompt(profile: dict, strategy: dict, language_name: str, max_questions: int) -> str:
    mode = strategy.get("input_mode", "speech")
    keys = f"Key mapping: {strategy['keypad_map']}." if strategy.get("keypad_map") else ""
    return SYSTEM_PROMPT_TEMPLATE.format(
        business_name=profile["business_name"],
        tone=profile["tone"],
        feedback_objective=profile["feedback_objective"],
        language_name=language_name,
        strategy_name=strategy["name"],
        dimension_type=strategy["dimension_type"],
        strategy_description=strategy["description"],
        sample_question=strategy["sample_question"],
        input_mode_line=INPUT_MODE_LINES.get(mode, INPUT_MODE_LINES["speech"]).format(keys=keys).strip(),
        max_questions=max_questions,
        call_seconds=CALL_DURATION_S,
        score_guidance=strategy["score_guidance"],
    )


_SPEECH_RE = re.compile(r'"speech"\s*:\s*"((?:[^"\\]|\\.)*)')

def _partial_speech(buf: str) -> str:
    """Best-effort extract of the (possibly still-growing) `speech` string from partial JSON."""
    m = _SPEECH_RE.search(buf)
    if not m:
        return ""
    try:
        return json.loads(f'"{m.group(1)}"')
    except json.JSONDecodeError:
        return ""

In [ ]:
def run_call(profile: dict, strategy: dict, language_name: str, language_code: str,
             fallback_closing: str, max_questions: int,
             stt: STTEngine, tts: TTSEngine) -> CallResult:
    system_prompt = build_system_prompt(profile, strategy, language_name, max_questions)
    chat = client.chats.create(
        model=TURN_MODEL,
        config=json_config(TURN_SCHEMA, TURN_MAX_TOKENS, system=system_prompt, temperature=0.4))

    result = CallResult()
    started = time.perf_counter()
    budget_used = 0.0          # seconds of real airtime consumed
    silence_strikes = 0
    turns = 0
    # Hint for the input engine: a real telephony leg needs to know whether to arm
    # DTMF capture for this question. It still accepts speech either way.
    expects_digits = strategy.get("input_mode") in ("keypad", "either")

    def remaining() -> float:
        return CALL_DURATION_S - (time.perf_counter() - started - budget_used)

    def send_turn(message: str) -> dict:
        """Stream one turn: speak `speech` as soon as it arrives, then return the parsed object.
        Falls back to a plain hang-up if the model output is unusable, instead of raising."""
        t0 = time.perf_counter()
        buf, spoken = "", ""
        try:
            for chunk in chat.send_message_stream(message):
                buf += chunk.text or ""
                partial = _partial_speech(buf)
                if len(partial) > len(spoken):
                    tts.synthesize(partial, language_code, final=False)
                    spoken = partial
            turn_json = json.loads(buf)
        except (json.JSONDecodeError, ValueError, TypeError):
            turn_json = {"speech": fallback_closing, "should_end": True, "end_reason": "error"}
        latency = time.perf_counter() - t0
        result.turn_latencies_s.append(latency)
        # Flush anything the stream held back (or the fallback text) and close the line.
        tts.synthesize(turn_json.get("speech", ""), language_code, final=True)
        if VERBOSE_LATENCY:
            print(f"    (turn latency {latency:.2f}s)")
        return turn_json

    def record(turn_json: dict) -> None:
        result.transcript.append({"role": "ai", "text": turn_json.get("speech", "")})
        if turn_json.get("score") is not None:
            result.score = turn_json["score"]
        if turn_json.get("sentiment"):
            result.sentiment = turn_json["sentiment"]

    def hard_close(reason: str) -> None:
        # No-LLM-call fallback, so a hard cutoff never waits on another round trip.
        tts.synthesize(fallback_closing, language_code, final=True)
        result.transcript.append({"role": "ai", "text": fallback_closing})
        result.end_reason = reason

    def finish(reason: str) -> CallResult:
        result.end_reason = result.end_reason or reason
        result.duration_s = time.perf_counter() - started - budget_used
        lat = result.turn_latencies_s
        avg = f"{sum(lat) / len(lat):.2f}s" if lat else "n/a"
        print(f"\n--- Call ended: {result.end_reason} | score={result.score} | "
              f"sentiment={result.sentiment} | airtime={result.duration_s:.1f}s | "
              f"turns={len(lat)} | avg turn {avg} ---")
        return result

    opening = send_turn("Begin the call now with your greeting and first question.")
    record(opening)
    if opening.get("should_end"):
        return finish(opening.get("end_reason") or "error")

    while True:
        if remaining() <= 1:
            hard_close("time_up")
            break
        if turns >= MAX_TURNS:
            hard_close("max_turns")
            break

        t_listen = time.perf_counter()
        customer = stt.listen(language_code, min(TURN_LISTEN_TIMEOUT_S, remaining()),
                              expect_digits=expects_digits)
        listened = time.perf_counter() - t_listen
        # Only the engine's real listening time counts against the 45s; the console
        # mock reports 0 so human typing does not blow the budget in a demo.
        budget_used += max(0.0, listened - min(stt.billable_seconds(), listened))
        turns += 1

        result.transcript.append({"role": "customer", "text": customer.as_transcript()})
        if customer.digits:
            result.keypresses.append(customer.digits)

        if customer.empty:
            silence_strikes += 1
            if silence_strikes > MAX_SILENCE_STRIKES:
                hard_close("silence")
                break
            reply = send_turn("<no response>")
            record(reply)
            if reply.get("should_end"):
                result.end_reason = reply.get("end_reason") or "silence"
                break
            continue

        silence_strikes = 0
        reply = send_turn(customer.as_model_message())
        record(reply)
        if reply.get("should_end"):
            result.end_reason = reply.get("end_reason") or "completed"
            break

    return finish("completed")

## 6. Run the Call

In [ ]:
call_result = run_call(business_profile, selected_strategy, LANG_NAME, LANG_CODE,
                       FALLBACK_CLOSING, MAX_QUESTIONS, stt_engine, tts_engine)
call_result.transcript

## 7. Extract Structured FeedbackTurn the raw transcript into a structured result. The scoring rule comes from the approach the model itself designed in Section 2, so this step adapts automatically to whatever scale was agreed on. The score the agent tracked live is passed in as a cross-check -- the summarizer is told to trust the transcript over it.

In [ ]:
SUMMARY_SCHEMA = types.Schema(
    type="OBJECT",
    properties={
        "score": types.Schema(type="NUMBER", nullable=True),
        "sentiment": types.Schema(type="STRING", enum=["positive", "neutral", "negative"]),
        "summary": types.Schema(type="STRING"),
        "key_points": types.Schema(type="ARRAY", items=types.Schema(type="STRING"), max_items=5),
        "answered": types.Schema(type="BOOLEAN"),
        "follow_up_needed": types.Schema(type="BOOLEAN"),
    },
    required=["score", "sentiment", "summary", "key_points", "answered", "follow_up_needed"])

SUMMARY_PROMPT = """Extract a structured result from this customer feedback call.

Approach used: {strategy_name} ({dimension_type}) -- {strategy_description}
How the customer answered: {input_mode}. {keypad_map}
Scoring rule: {score_guidance}
Call ended because: {end_reason}
Score the agent tracked live (may be wrong -- the transcript wins): {live_score}
Keys the customer pressed, in order: {keypresses}

Transcript:
{transcript}

Rules:
- This was a phone call. "[pressed 4]" in the transcript means the customer pressed 4 on their keypad -- that IS their answer, as valid as speaking it, so read it through the key mapping above.
- Use ONLY what the customer actually said or pressed. Never infer a number they did not give: if they gave no usable rating, score is null.
- summary: 1-2 sentences, factual, no praise or padding.
- key_points: at most 5 short phrases, each a distinct piece of feedback about the business. Do not restate the score, and exclude off-topic remarks and questions the customer asked. Empty list if they gave no substantive feedback.
- answered: true only if the customer gave a real answer to at least one feedback question.
- follow_up_needed: true if the customer voiced ANY dissatisfaction, complaint, or unresolved issue (a long wait, a mistake, cold food), or asked for a human -- even alongside praise and even with a high score.
"""

def summarize_feedback(result: CallResult, strategy: dict) -> dict:
    transcript_text = "\n".join(f"{t['role']}: {t['text']}" for t in result.transcript)
    prompt = SUMMARY_PROMPT.format(
        strategy_name=strategy["name"],
        dimension_type=strategy["dimension_type"],
        strategy_description=strategy["description"],
        input_mode=MODE_LABEL.get(strategy.get("input_mode", "speech"), "spoken answer"),
        keypad_map=f"Key mapping: {strategy['keypad_map']}." if strategy.get("keypad_map") else "",
        score_guidance=strategy["score_guidance"],
        end_reason=result.end_reason,
        live_score=result.score,
        keypresses=", ".join(result.keypresses) or "none",
        transcript=transcript_text,
    )
    data = call_json(PLANNING_MODEL, prompt, SUMMARY_SCHEMA, max_tokens=1024)
    data["call_ended_reason"] = result.end_reason
    data["dimension_type"] = strategy["dimension_type"]
    return data

feedback_summary = summarize_feedback(call_result, selected_strategy)
feedback_summary

## 8. Listen to the Call

Both sides were recorded into one timeline as they spoke -- the agent's TTS output and, depending
on the mode, either your microphone or the synthetic caller's voice. This writes it to a WAV next
to the notebook and embeds a player, which is also the only way to hear the call on Colab, where
there is no local audio device.

If the recording sounds like it has gaps, that is the honest picture: those are the TTS and STT
round trips the customer would experience as dead air, and they are exactly what a production
deployment removes by moving to a streaming, full-duplex telephony leg.

In [ ]:
from IPython.display import Audio, display

speaker.close()
tts_engine.close()

RECORDING_PATH = "call_recording.wav"
wav = call_recorder.wav_bytes()
with open(RECORDING_PATH, "wb") as f:
    f.write(wav)

print(f"Recorded {call_recorder.seconds:.1f}s of audio -> {RECORDING_PATH} ({len(wav) / 1024:.0f} KB)")
lat = call_result.turn_latencies_s
if lat:
    print(f"Turn model latency: min {min(lat):.2f}s | avg {sum(lat) / len(lat):.2f}s | max {max(lat):.2f}s")
print(f"Call airtime {call_result.duration_s:.1f}s against a {CALL_DURATION_S}s budget "
      f"({'within' if call_result.duration_s <= CALL_DURATION_S else 'over'})")

display(Audio(wav, rate=call_recorder.rate))